# Data augmentation

_Deep Learning_

**Make more training images by flipping, rotating, and cropping the ones you have.**

More data usually means a better model. But collecting it is expensive.

     Data augmentation creates new training examples from existing ones with simple changes: flip, rotate, crop, shift colors.

---

This notebook is a practice scaffold for the **Data augmentation** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Build an augmentation pipeline

`transforms.Compose` chains several random image transforms into one callable. Each transform leaves the label unchanged but alters the pixels: a horizontal flip, a small rotation, a zoom-and-crop, and a brightness tweak. Stacking them means every call produces a fresh, plausible variant of the input image.

In [ ]:
import torch
from torchvision import transforms

augment = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),              # mirror left-right
    transforms.RandomRotation(degrees=15),               # tilt a little
    transforms.RandomResizedCrop(32, scale=(0.8, 1.0)),  # zoom/crop
    transforms.ColorJitter(brightness=0.2),              # tweak brightness
])

### Step 2 — Apply it twice to one image

We make a fake 3-channel 40x40 image and run the pipeline on it twice. Because the transforms are random, the two results differ even though they came from the *same* source image — that's exactly how augmentation multiplies a dataset. Note the output is cropped to 32x32 by `RandomResizedCrop`.

In [ ]:
# a fake 3-channel 40x40 image as a tensor
img = torch.rand(3, 40, 40)

# each call yields a different random variant of the same image
v1 = augment(img)
v2 = augment(img)

print("augmented shape:", tuple(v1.shape))      # (3, 32, 32)
print("two variants differ:", not torch.equal(v1, v2))

## Visualize the data & results

_On real digits seen at an angle, does training-time augmentation actually improve accuracy?_

### Step 1 — Split the digits and rotate the test set

We split the digit images into a training set and a held-out test set, then deliberately **rotate every test image by 18 degrees**. This simulates seeing digits at an angle the model was never trained on, so we can measure whether augmentation helps the model cope with that shift.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from scipy.ndimage import rotate, shift

digits = load_digits()
Xtr, Xte, ytr, yte = train_test_split(
    digits.images, digits.target,
    test_size=0.3, random_state=0,
    stratify=digits.target,
)

# rotate the test digits so they arrive at an angle the model never saw
Xte_rot = np.array([rotate(im, 18, reshape=False) for im in Xte])

### Step 2 — Define a train-and-score helper

This helper takes a list of augmentations to apply. For each one it makes a transformed copy of the training images (rotations of various angles, or a 1-pixel shift), stacks all the copies together with the originals, trains a small network, and returns its accuracy on the rotated test set. More augmentations means more, more-varied training data.

In [ ]:
def accuracy(augs):
    imgs = [Xtr]                       # start with the original training images
    labs = [ytr]
    ops = {"rot": 15, "rotn": -15, "rot2": 20}   # named rotation angles

    for a in augs:
        if a in ops:
            rotated = np.array([rotate(im, ops[a], reshape=False) for im in Xtr])
            imgs.append(rotated)
        else:
            shifted = np.array([shift(im, [1, 1]) for im in Xtr])
            imgs.append(shifted)
        labs.append(ytr)               # the label is unchanged by augmentation

    # flatten each image set to rows and scale pixels to 0-1
    X = np.vstack([batch.reshape(len(batch), -1) for batch in imgs]) / 16.0
    Y = np.concatenate(labs)

    model = MLPClassifier(
        hidden_layer_sizes=(50,), max_iter=400,
        random_state=0, alpha=1e-3,
    ).fit(X, Y)

    Xte_flat = Xte_rot.reshape(len(Xte_rot), -1) / 16.0
    return accuracy_score(yte, model.predict(Xte_flat))

### Step 3 — Compare accuracy as augmentations stack up

We run the helper five times, each adding one more augmentation, and plot the resulting accuracy on the rotated test set. The bars should climb as more rotation/shift variety is added — concrete evidence that augmentation makes the model robust to digits it sees at an angle.

In [ ]:
levels = [
    [],
    ["rot"],
    ["rot", "rotn"],
    ["rot", "rotn", "rot2"],
    ["rot", "rotn", "rot2", "shift"],
]
labels = ["no aug", "+rotate15", "+rotate-15", "+rotate20", "+shift (all)"]
colors = ["#ff7b72", "#ffb454", "#ffb454", "#4ea1ff", "#7ee787"]

accs = [accuracy(level) for level in levels]

plt.bar(labels, accs, color=colors)
plt.title("Real accuracy on rotated test digits as augmentations are added")
plt.ylim(0.6, 1.0)
plt.show()